# Previsione Polidramnios

Carico il modello addestrato (`train.ipynb` -> `models/modello_polidramnios.joblib`) e
lo uso per prevedere il Polidramnios di una singola paziente a partire dai suoi dati.

## 1. Caricamento del modello

Il pacchetto `joblib` contiene la **pipeline completa** (preprocessore + classificatore) e i
metadati (`features`, `target`, `threshold`). La pipeline applica da sola imputazione e scaling.

In [1]:
from pathlib import Path
import joblib
import pandas as pd

def carica_modello(percorso='../models/modello_polidramnios.joblib'):
    """Carica il pacchetto joblib salvato dal training (pipeline + metadati)."""
    percorso = Path(percorso)
    if not percorso.exists():
        raise FileNotFoundError(f'Modello non trovato: {percorso}. Esegui prima train.ipynb.')
    return joblib.load(percorso)

pacchetto = carica_modello()
print('Feature attese:', pacchetto['features'])
print('Soglia di decisione:', pacchetto['threshold'])

Feature attese: ['Eta_anni', 'Numero_Gravidanze_Pregresse', 'Numero_Tagli_Cesarei_Pregressi', 'Pressione_Diastolica_mmHg', 'Insulina_Sierica_2ore', 'Indice_Massa_Corporea', 'Diabete_Gestazionale', 'Diabete_Pregravidico']
Soglia di decisione: 0.5


## 2. Funzione di previsione

Prende i dati della futura mamma, costruisce una riga nell'ordine atteso dal modello e
restituisce la **previsione** (0/1, in base alla soglia) e la **probabilità** di Polidramnios.

In [2]:
def prevedi_polidramnios(eta, gravidanze_pregresse, tagli_cesarei_pregressi,
                         pressione_diastolica, insulina_sierica_2ore, indice_massa_corporea,
                         diabete_gestazionale, diabete_pregravidico, pacchetto=pacchetto):
    """Previsione per una singola paziente. Ritorna (previsione 0/1, probabilita')."""
    valori = {
        'Eta_anni': eta,
        'Numero_Gravidanze_Pregresse': gravidanze_pregresse,
        'Numero_Tagli_Cesarei_Pregressi': tagli_cesarei_pregressi,
        'Pressione_Diastolica_mmHg': pressione_diastolica,
        'Insulina_Sierica_2ore': insulina_sierica_2ore,
        'Indice_Massa_Corporea': indice_massa_corporea,
        'Diabete_Gestazionale': diabete_gestazionale,
        'Diabete_Pregravidico': diabete_pregravidico,
    }
    # Una riga, colonne nell'ordine atteso dalla pipeline.
    X = pd.DataFrame([valori])[pacchetto['features']]
    prob = float(pacchetto['pipeline'].predict_proba(X)[0, 1])
    previsione = int(prob >= pacchetto['threshold'])
    return previsione, prob

## 3. Esempio d'uso

In [ ]:
pred, prob = prevedi_polidramnios(
    eta=34, gravidanze_pregresse=2, tagli_cesarei_pregressi=1,
    pressione_diastolica=78, insulina_sierica_2ore=140, indice_massa_corporea=31.5,
    diabete_gestazionale=1, diabete_pregravidico=0)

print(f'Previsione: {pred}  (1 = Polidramnios, 0 = no)')
print(f'Probabilita di Polidramnios: {prob:.1%}')

Previsione: 1  (1 = Polidramnios, 0 = no)
Probabilita di Polidramnios: 89.7%
